# Ch 23. KV キャッシュ — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: KV キャッシュの有無で 100 トークン生成時間を $n=128,512,2048$ について比較せよ。

### 解答

キャッシュなしでは各 decode で prefix 全体の K,V を再計算し、仕事量は概ね $\sum_{t=n}^{n+99}t$ に比例する。ありでは新 token のみで 100 に比例する。時間はハードウェア依存なので厳密な仕事量代理値と増加率を示す。


In [ ]:
import time
import numpy as np

# Reduced autoregressive projection benchmark: recompute the prefix or append one cached row.
rng=np.random.default_rng(2301); width=64; generated=100; W=rng.normal(size=(width,width)); results={}
for initial in (128,512,2048):
    tokens=rng.normal(size=(initial+generated,width))
    start=time.perf_counter();
    for step in range(generated): tokens[:initial+step+1]@W
    no_cache=time.perf_counter()-start
    start=time.perf_counter(); cache=tokens[:initial]@W
    for step in range(generated): cache=np.vstack((cache,tokens[initial+step:initial+step+1]@W))
    cached=time.perf_counter()-start
    results[initial]={"no_cache_ms":1000*no_cache,"cache_ms":1000*cached,"speedup":no_cache/cached}
assert all(r["speedup"]>1 for r in results.values())
print({n:{k:round(v,3) for k,v in r.items()} for n,r in results.items()})


## 問題 2

**問題**: LLaMA-7B の 32K コンテキストにおける KV キャッシュメモリを計算せよ。

### 解答

LLaMA-7B は 32 層、32 KV ヘッド、ヘッド次元 128。FP16 では $2\times32\times32\times128\times32768\times2$ bytes $=16$ GiB。最初の 2 は K,V、最後は 1 要素の byte 数である。


In [ ]:
layers,kv_heads,head_dim,context,batch,bytes_per_value=32,32,128,32768,1,2
bytes_used=2*layers*kv_heads*head_dim*context*batch*bytes_per_value
gib=bytes_used/1024**3
assert bytes_used==17_179_869_184 and gib==16.0
print({"layers":layers,"kv_heads":kv_heads,"head_dim":head_dim,"context":context,"batch":batch,
       "dtype":"fp16","KV_cache_bytes":bytes_used,"KV_cache_GiB":gib})


## 問題 3

**問題**: GQA が KV キャッシュメモリをどれだけ節約するか、LLaMA-7B（32 ヘッド）と LLaMA-70B（8 KV ヘッド）で比較せよ。

### 解答

同じ層数・ヘッド次元・長さならキャッシュは KV ヘッド数に線形である。32 から 8 は 1/4、75% 節約。実際の 7B と 70B は層数が異なるため総量比較には各層数も含める。
### 解説

指定された2モデル全体の比較では層数も含める必要がある。7Bは $32\times32=1024$ layer-heads、70Bは $80\times8=640$ なので、context、head dimension、dtypeが同じなら70Bの総KV cacheは7Bの62.5%、つまり37.5%削減となる。別途、80層architectureを固定してKV headだけを32から8へ変える仮想比較では75%削減となる。


In [ ]:
# Named-model total includes each architecture's layer count; fixed-architecture isolates head sharing.
batch,context,head_dim,bytes_per_value=1,32768,128,2
def kv_bytes(layers,heads): return 2*batch*layers*heads*context*head_dim*bytes_per_value
llama_7b=kv_bytes(32,32); llama_70b=kv_bytes(80,8)
named_saving=1-llama_70b/llama_7b
fixed_mha=kv_bytes(80,32); fixed_gqa=kv_bytes(80,8); fixed_saving=1-fixed_gqa/fixed_mha
assert named_saving==0.375 and fixed_saving==0.75
print({"named_models":{"LLaMA-7B_GiB":llama_7b/1024**3,"LLaMA-70B_GiB":llama_70b/1024**3,
       "70B_vs_7B_saving":named_saving},"fixed_80_layer_architecture":{"32_to_8_KV_heads_saving":fixed_saving}})


## 問題 4

**問題**: Continuous Batching のスループット向上をシミュレーションで示せ。

### 解答

静的 batching は最長要求終了まで空き slot を残す。continuous batching は完了直後に新要求を投入し active slot 時間を増やす。以下の離散 event simulation で同一要求列の総時間と throughput を決定的に比較する。


In [ ]:
import heapq

lengths=[9,2,7,3,8,4,6,5,2,9,3,7]; slots=3
static_time=sum(max(lengths[i:i+slots]) for i in range(0,len(lengths),slots))
workers=[0]*slots
for length in lengths:
    earliest=heapq.heappop(workers); heapq.heappush(workers,earliest+length)
continuous_time=max(workers)
tokens=sum(lengths); static_throughput=tokens/static_time; continuous_throughput=tokens/continuous_time
assert continuous_throughput>static_throughput
print({"requests":lengths,"static_ticks":static_time,"continuous_ticks":continuous_time,
       "static_tokens_per_tick":round(static_throughput,3),"continuous_tokens_per_tick":round(continuous_throughput,3),
       "throughput_gain":round(continuous_throughput/static_throughput,3)})


## 問題 5

**問題**: PagedAttention がメモリ断片化をどのように解決するか説明せよ。

### 解答

PagedAttention は論理的に連続な KV を固定長物理 block に分け page table で写像する。最大長の連続領域を事前確保せず外部断片化を減らし、内部浪費は各系列の末尾 1 block 未満となる。block 共有で prefix 再利用も可能になる。


In [ ]:
import math

requests=[17,33,65,7]; block=16
contiguous_reserved=[64,64,128,32]
paged_reserved=[math.ceil(n/block)*block for n in requests]
contiguous_waste=sum(contiguous_reserved)-sum(requests); paged_waste=sum(paged_reserved)-sum(requests)
page_table={request:list(range(sum(paged_reserved[:i])//block,sum(paged_reserved[:i+1])//block)) for i,request in enumerate(requests)}
assert paged_waste < contiguous_waste and all(waste < block for waste in [r-n for r,n in zip(paged_reserved,requests)])
print({"request_tokens":requests,"contiguous_waste":contiguous_waste,"paged_waste":paged_waste,"page_table":page_table})
